In [6]:
import zipfile
zip = zipfile.ZipFile("archive (7).zip",'r')
zip.extractall("/content")
zip.close()

In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [10]:
import os
print(os.listdir())

['.git', '.ipynb_checkpoints', 'age-gender-revised.ipynb', 'archive (6).zip', 'archive (7).zip', 'backpropagation-classification.ipynb', 'backpropagation-regression.ipynb', 'Batch-vs-stochastic-GD.ipynb', 'batch_norm_example.ipynb', 'cat-vs-dog.ipynb', 'Copy_of_zero_initialization_relu.ipynb', 'dataset', 'dropout-classification-example.ipynb', 'dropout-notebook.ipynb', 'early-stopping.ipynb', 'feature-scaling.ipynb', 'functional-api-demo.ipynb', 'functional-api-multiple-input.ipynb', 'keras-padding-demo.ipynb', 'keras-pooling-demo.ipynb', 'kohli.jpg', 'perceptron-demo.ipynb', 'perceptron-trick.ipynb', 'placement (2).csv', 'regularization.ipynb', 'Social_Network_Ads (2).csv', 'transfer-learning-feature-extraction(data-augmentation).ipynb', 'transfer-learning-feature-extraction(without-data-augmentation).ipynb.ipynb', 'transfer-learning-finetuning.ipynb.ipynb', 'vanishing-gradient.ipynb', 'vanishing-gradient1.ipynb', 'visualising-cnn.ipynb', 'weight-initialization.ipynb']


In [8]:
folder_path = 'archive (7).zip'


In [9]:
age=[]
gender=[]
img_path=[]
for file in os.listdir(folder_path):
  age.append(int(file.split('_')[0]))
  gender.append(int(file.split('_')[1]))
  img_path.append(file)

NotADirectoryError: [WinError 267] The directory name is invalid: 'archive (7).zip'

In [ ]:
len(age)

In [ ]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [ ]:
train_df.shape

In [ ]:
test_df.shape

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='multi_output')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                  class_mode='multi_output')

In [ ]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [ ]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

In [ ]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

resnet.trainable=False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [ ]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [ ]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':1,'gender':99})

In [ ]:
model.fit(train_generator, batch_size=32, epochs=10, validation_data=test_generator)